# Stage 1 — GPU benchmark + probing sweep

Runs `scripts/run_logit_benchmarks.py` and `scripts/run_probing.py` on a Colab GPU. Results are written to Google Drive so they persist across disconnects; the scripts auto-resume from per-`(model, benchmark)` JSON checkpoints if the runtime is killed.

**Setup (once):** Runtime → Change runtime type → GPU (A100 recommended). Sidebar key icon → add Colab Secrets `HF_TOKEN` and `WANDB_API_KEY` with notebook access enabled.

**Useful flags on `run_logit_benchmarks.py`:**
- `--family {mistral,llama,qwen,gemma}` — restrict to one family
- `--models <id1> <id2> ...` — restrict to specific HF model IDs (overrides `--family`)
- `--variant {base,instruct}` — restrict to one side of each pair
- `--benchmarks crows_pairs stereoset bbq iat` — subset of benchmarks
- `--prompt-modes raw instruct` — prompt conditions
- `--languages en fr es de pt it` — multilingual CrowS-Pairs (other benchmarks ignore this)

**Useful flags on `run_probing.py`:**
- `--family`, `--variant` — same as above
- `--mask-keywords` — Audit-3 controlled probe: excludes demographic surface forms from the activation pool. Writes to `results/activations_masked/` and `results/probe_results_masked/` so the unmasked baseline is preserved for comparison.

In [ ]:
GITHUB_REPO = 'https://github.com/korneli777/llm-bias-eval.git'
FAMILY = 'mistral'  # mistral | llama | qwen | gemma
DRIVE_DIR = '/content/drive/MyDrive/llm-bias-eval'

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata
from huggingface_hub import login

REPO_DIR = '/content/llm-bias-eval'

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

results_link = os.path.join(REPO_DIR, 'results')
if os.path.lexists(results_link):
    (os.unlink if os.path.islink(results_link) else shutil.rmtree)(results_link)
os.symlink(f'{DRIVE_DIR}/results', results_link)

subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'llm-bias-eval'

# Explicit HF login persists the token to disk so subprocesses pick it up
# (env var alone doesn't always propagate to !python ... cells).
login(token=os.environ['HF_TOKEN'])

In [ ]:
!python scripts/run_logit_benchmarks.py --family {FAMILY}

In [ ]:
# Multilingual CrowS-Pairs (skip if running English-only).
# Other benchmarks ignore --languages, so passing crows_pairs explicitly avoids
# re-running stereoset/bbq/iat unnecessarily.
!python scripts/run_logit_benchmarks.py --family {FAMILY} \
    --benchmarks crows_pairs --languages fr es de pt it

In [ ]:
!python scripts/run_probing.py --family {FAMILY}

In [ ]:
# Audit-3 controlled probe: re-extract activations with demographic-keyword
# tokens masked out of the pool. Writes to results/{activations,probe_results}_masked/
# so the original probe results are untouched. Skip if you only need the
# unmasked baseline.
!python scripts/run_probing.py --family {FAMILY} --mask-keywords